# f6_m01d_shapash.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M01d — Dashboard Shapash |

---

## 🎯 Qué hace

Dashboard interactivo de interpretabilidad SHAP sobre el conjunto de test.
Construido con Plotly — sin dependencia de servidor, embebible en HTML.
Incluye 4 visualizaciones interactivas: waterfall individual (TP/FN/FP),
beeswarm filtrable por tipo de clasificación, tabla de los 200 alumnos de
mayor riesgo e importancia SHAP comparativa entre grupos. Trabaja sobre el
modelo ganador, leído dinámicamente del sistema.

## 📋 Requisitos

- `results/fase6/shap_global_ganador.pkl` — valores SHAP del ganador (generado por f6_m01a)
- `results/fase6/shap_global_ganador_meta.json` — metadatos de validación del cache
- `results/fase6/shap_importancia_comparativa.parquet` — rankings unificados (generado por f6_m01a)
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/06_evaluacion/metricas_modelo.json` — define el modelo ganador (generado por f6_m00_preparacion)
- `data/05_modelado/models/` — modelo ganador (según el JSON)
- `src/config_entorno.py` — NOMBRES_LEGIBLES_FEATURES, RAMAS_NOMBRES
- `src/html/wilcoxon_block.py` — bloque Wilcoxon reutilizable para el HTML

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase6/m01d_shapash.html` | Dashboard interactivo completo |
| `results/fase6/shapash_dashboard.html` | Copia de respaldo del dashboard |

## 🔄 Flujo

```
shap_global_ganador.pkl + shap_importancia_comparativa.parquet (m01a)
    ↓ Validar cache SHAP
    ↓ Waterfall TP/FN/FP (Plotly)
    ↓ Beeswarm filtrable por tipo (Plotly)
    ↓ Tabla top 200 alumnos de mayor riesgo (Plotly)
    ↓ Importancia SHAP por grupo abandonan/no abandonan (Plotly)
    → docs/html/fase6/m01d_shapash.html + copia de respaldo en results/fase6/
```

## ➡️ Siguiente

`f6_m02a_lime.ipynb` — explicaciones locales con LIME


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/.
# RUTA_JSON: modelo ganador leído dinámicamente (sistema v2).
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Detección robusta de ROOT subiendo niveles hasta encontrar src/
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_HTML    = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

RUTA_JSON = ROOT / 'data' / '06_evaluacion' / 'metricas_modelo.json'

print(f'ROOT:       {ROOT}')
print(f'DIR_RESULTS:{DIR_RESULTS}')
print(f'RUTA_JSON:  {RUTA_JSON}')

ROOT:       c:\PRUEBAS\AU_UJI_v2_RUTA_B
DIR_RESULTS:c:\PRUEBAS\AU_UJI_v2_RUTA_B\results\fase6
RUTA_JSON:  c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\06_evaluacion\metricas_modelo.json


In [2]:
# ============================================================
# CELDA 2: IMPORTS Y DICCIONARIO DE NOMBRES LEGIBLES
# ============================================================
import json
import hashlib
import base64
import numpy as np
import pandas as pd
import joblib
import shap
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from src.html.render import render_pagina_desde_fichero

# Nombres legibles desde config_entorno — fuente única de verdad
from src.config_entorno import NOMBRES_LEGIBLES_FEATURES as NOMBRES_LEGIBLES, RAMAS_NOMBRES


def nombre_legible(f: str) -> str:
    return NOMBRES_LEGIBLES.get(f, f.replace('_', ' '))

print('Imports OK.')

Imports OK.


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS, MODELO Y SHAP CON VALIDACIÓN
# ============================================================
# Carga los .pkl generados por m01a (modelo ganador + shap_ganador) — no
# recalcula. Valida shape y metadatos antes de continuar.
# ============================================================
import json as _json

def _hash_df(df: pd.DataFrame, n: int = 500) -> str:
    muestra = df.iloc[:n].values.tobytes()
    return hashlib.md5(muestra).hexdigest()[:12]

X_test = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()

feature_names  = X_test.columns.tolist()
feature_labels = [nombre_legible(f) for f in feature_names]

# Cargar shap_importancia_comparativa (generado por m01a)
RUTA_IMP = DIR_RESULTS / 'shap_importancia_comparativa.parquet'
if not RUTA_IMP.exists():
    raise FileNotFoundError(
        'No se encontró shap_importancia_comparativa.parquet.\n'
        'Ejecuta primero f6_m01a_shap_global.ipynb.'
    )
df_imp = pd.read_parquet(RUTA_IMP)

# --- Leer modelo ganador del JSON dinámico ---
assert RUTA_JSON.exists(), f'❌ No encontrado: {RUTA_JSON}'
with open(RUTA_JSON, encoding='utf-8') as f:
    meta_json = _json.load(f)

nombre_ganador_pkl = meta_json['modelo_pkl']
nombre_ganador     = meta_json['modelo_nombre']
familia_ganador    = meta_json['modelo_familia']

modelo_ganador = joblib.load(DIR_MODELS / nombre_ganador_pkl)
ganador_raw    = modelo_ganador.named_steps['model']

y_prob = modelo_ganador.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)
y_true = y_test.values.ravel()

# Cargar y validar SHAP
RUTA_SHAP_GANADOR = DIR_RESULTS / 'shap_global_ganador.pkl'
RUTA_META_GANADOR = DIR_RESULTS / 'shap_global_ganador_meta.json'

if not RUTA_SHAP_GANADOR.exists():
    raise FileNotFoundError(
        'No se encontró shap_global_ganador.pkl.\n'
        'Ejecuta primero f6_m01a_shap_global.ipynb.'
    )

if RUTA_META_GANADOR.exists():
    meta = json.loads(RUTA_META_GANADOR.read_text())
    meta_esp = {
        'n_obs':      int(len(X_test)),
        'n_features': int(X_test.shape[1]),
        'hash_xtest': _hash_df(X_test),
    }
    if meta != meta_esp:
        raise ValueError(
            f'Cache SHAP desactualizado.\nGuardado: {meta}\nEsperado: {meta_esp}\n'
            'Ejecuta f6_m01a_shap_global.ipynb para regenerar.'
        )

shap_ganador = joblib.load(RUTA_SHAP_GANADOR)

# Extraer matriz de valores SHAP
shap_matrix = shap_ganador.values
if shap_matrix.ndim == 3:
    shap_matrix = shap_matrix[:, :, 1]

assert shap_matrix.shape == (len(X_test), len(feature_names)), (
    f'Shape SHAP {shap_matrix.shape} no coincide con '
    f'X_test ({len(X_test)}, {len(feature_names)})'
)

# Tipo de clasificación por observación
tipo_cls = np.where(
    (y_true==1)&(y_pred==1), 'TP',
    np.where((y_true==0)&(y_pred==0), 'TN',
    np.where((y_true==0)&(y_pred==1), 'FP', 'FN'))
)

print(f'📌 Modelo ganador: {nombre_ganador} ({nombre_ganador_pkl})')
print(f'X_test:      {X_test.shape}')
print(f'SHAP matrix: {shap_matrix.shape}')
print(f'Features:    {len(feature_names)}')
print(f'TP={(tipo_cls=="TP").sum()} TN={(tipo_cls=="TN").sum()} '
      f'FP={(tipo_cls=="FP").sum()} FN={(tipo_cls=="FN").sum()}')
print('✅ Datos cargados y validados.')

📌 Modelo ganador: LightGBM (LightGBM__none.pkl)
X_test:      (6725, 27)
SHAP matrix: (6725, 27)
Features:    27
TP=1583 TN=4509 FP=249 FN=384
✅ Datos cargados y validados.


In [4]:
# ============================================================
# CELDA 4: GRÁFICO 1 — WATERFALL INDIVIDUAL (SHAP, PLOTLY)
# ============================================================
# Muestra cómo cada variable empuja la predicción para un alumno concreto.
# Se generan 3 ejemplos: TP, FN y FP.
# ============================================================
def waterfall_alumno(idx: int, titulo: str) -> go.Figure:
    shap_vals = shap_matrix[idx]
    feat_vals = X_test.iloc[idx]
    base_value = shap_ganador.base_values[idx] if hasattr(shap_ganador, 'base_values') else 0

    # Top 10 por valor absoluto
    orden  = np.argsort(np.abs(shap_vals))[::-1][:10]
    sv     = shap_vals[orden]
    fnames = [f'{nombre_legible(feature_names[i])} = {feat_vals.iloc[i]:.2g}'
              for i in orden]
    colores = ['#e53e3e' if v > 0 else '#3182ce' for v in sv]

    fig = go.Figure(go.Bar(
        x=sv[::-1], y=fnames[::-1],
        orientation='h',
        marker_color=colores[::-1],
        text=[f'{v:+.3f}' for v in sv[::-1]],
        textposition='outside',
        hovertemplate='%{y}<br>SHAP: %{x:.4f}<extra></extra>',
    ))
    prob = y_prob[idx]
    real = 'Abandona' if y_true[idx]==1 else 'No abandona'
    pred = 'Abandona' if y_pred[idx]==1 else 'No abandona'
    fig.update_layout(
        title=dict(
            text=f'{titulo}<br><sup>Prob: {prob:.3f} · Real: {real} · '
                 f'Pred: {pred} · Tipo: {tipo_cls[idx]}</sup>',
            font=dict(size=13)
        ),
        xaxis=dict(title='Contribución SHAP (rojo = aumenta riesgo, azul = reduce)'),
        height=420, width=780,
        plot_bgcolor='#f7fafc', paper_bgcolor='white',
        margin=dict(l=220, r=80, t=80, b=40),
    )
    return fig

idx_tp = int(np.where(tipo_cls=='TP')[0][0])
idx_fn = int(np.where(tipo_cls=='FN')[0][0])
idx_fp = int(np.where(tipo_cls=='FP')[0][0])

fig_wf_tp = waterfall_alumno(idx_tp, 'Alumno detectado correctamente (TP)')
fig_wf_fn = waterfall_alumno(idx_fn, 'Alumno no detectado — falso negativo (FN)')
fig_wf_fp = waterfall_alumno(idx_fp, 'Falsa alarma — falso positivo (FP)')

fig_wf_tp.show()
fig_wf_fn.show()
fig_wf_fp.show()
print('✅ Waterfalls generados.')

✅ Waterfalls generados.


In [5]:
# ============================================================
# CELDA 5: GRÁFICO 2 — BEESWARM FILTRABLE POR TIPO
# Cada punto = un alumno. Eje X = valor SHAP.
# Botones para filtrar por TP / FN / FP / TN / Todos.
# Muestra de 800 obs para no saturar el HTML.
# ============================================================
imp_media  = np.abs(shap_matrix).mean(axis=0)
top12_idx  = np.argsort(imp_media)[::-1][:12]
top12_names = [nombre_legible(feature_names[i]) for i in top12_idx]

np.random.seed(42)
sample_idx  = np.random.choice(len(X_test), min(800, len(X_test)), replace=False)

tipos_orden  = ['Todos', 'TP', 'FN', 'FP', 'TN']
colores_tipo = {'TP':'#27ae60','FN':'#e74c3c','FP':'#e67e22','TN':'#95a5a6','Todos':'#3182ce'}

traces = []
for tipo in tipos_orden:
    mask = np.ones(len(sample_idx), dtype=bool) if tipo == 'Todos' else tipo_cls[sample_idx] == tipo
    si   = sample_idx[mask]
    for j, fi in enumerate(top12_idx):
        fname = nombre_legible(feature_names[fi])
        sv    = shap_matrix[si, fi]
        traces.append(go.Box(
            x=sv, name=fname,
            orientation='h',
            marker=dict(color=colores_tipo[tipo], opacity=0.5, size=4),
            boxpoints='all', jitter=0.4, pointpos=0,
            visible=(tipo == 'Todos'),
            legendgroup=tipo,
            showlegend=(j==0),
            hovertemplate=f'<b>{fname}</b><br>SHAP: %{{x:.4f}}<extra>{tipo}</extra>',
        ))

n_feat  = len(top12_idx)
n_tipos = len(tipos_orden)
buttons = []
for ti, tipo in enumerate(tipos_orden):
    vis = [False] * (n_feat * n_tipos)
    for j in range(n_feat):
        vis[ti * n_feat + j] = True
    n_tipo = (tipo_cls==tipo).sum() if tipo != 'Todos' else len(tipo_cls)
    buttons.append(dict(
        label=tipo, method='update',
        args=[{'visible': vis},
              {'title': f'SHAP Beeswarm — {tipo} ({n_tipo} alumnos)'}]
    ))

fig_bee = go.Figure(data=traces)
fig_bee.update_layout(
    title='SHAP Beeswarm — Top 12 variables (filtrable por tipo de clasificación)',
    xaxis=dict(title='Valor SHAP (+ = mayor riesgo abandono)',
               zeroline=True, zerolinecolor='#718096', zerolinewidth=1),
    height=560, width=820,
    plot_bgcolor='#f7fafc', paper_bgcolor='white',
    updatemenus=[dict(
        type='buttons', direction='right',
        x=0.0, y=1.12, xanchor='left',
        buttons=buttons,
        bgcolor='#edf2f7', bordercolor='#e2e8f0',
        font=dict(size=11),
    )],
    margin=dict(l=200, r=40, t=100, b=40),
)
fig_bee.show()
print('✅ Beeswarm filtrable generado.')

✅ Beeswarm filtrable generado.


In [6]:
# ============================================================
# CELDA 6: GRÁFICO 3 — TABLA INTERACTIVA DE ALUMNOS
# Top 200 alumnos con mayor probabilidad predicha.
# Ordenable por columna. Coloreada por tipo de clasificación.
# ============================================================
top200_idx = np.argsort(y_prob)[::-1][:200]

df_tabla = pd.DataFrame({
    'Índice':         top200_idx,
    'Prob. abandono': y_prob[top200_idx].round(3),
    'Real':           ['Abandona' if v==1 else 'No abandona' for v in y_true[top200_idx]],
    'Predicción':     ['Abandona' if v==1 else 'No abandona' for v in y_pred[top200_idx]],
    'Tipo':           tipo_cls[top200_idx],
})

# Añadir top 5 features
top5_feat = [feature_names[i] for i in top12_idx[:5]]
for f in top5_feat:
    df_tabla[nombre_legible(f)] = X_test.iloc[top200_idx][f].values.round(2)

colores_fila = {'TP':'#f0fff4','TN':'#ebf8ff','FP':'#fffbeb','FN':'#fff5f5'}
fill_colors  = [[colores_fila.get(t, 'white') for t in df_tabla['Tipo']]]

fig_tabla = go.Figure(go.Table(
    header=dict(
        values=list(df_tabla.columns),
        fill_color='#edf2f7',
        font=dict(size=11, color='#2d3748'),
        align='left', height=32,
    ),
    cells=dict(
        values=[df_tabla[c] for c in df_tabla.columns],
        fill_color=['#f7fafc'] + fill_colors * (len(df_tabla.columns)-1),
        font=dict(size=11, color='#2d3748'),
        align='left', height=26,
    )
))
fig_tabla.update_layout(
    title='Top 200 alumnos con mayor riesgo predicho',
    height=520, width=900,
    margin=dict(t=60, b=10, l=10, r=10),
)
fig_tabla.show()
print('✅ Tabla interactiva generada.')

✅ Tabla interactiva generada.


In [7]:
# ============================================================
# CELDA 7: GRÁFICO 4 — IMPORTANCIA SHAP POR GRUPO
# Compara importancia media entre alumnos que abandonan
# vs no abandonan. Revela qué variables discriminan más.
# ============================================================
mask_ab  = y_true == 1
mask_nab = y_true == 0

imp_ab  = np.abs(shap_matrix[mask_ab]).mean(axis=0)
imp_nab = np.abs(shap_matrix[mask_nab]).mean(axis=0)

diff    = np.abs(imp_ab - imp_nab)
orden_diff  = np.argsort(diff)[::-1][:12]
labels_diff = [nombre_legible(feature_names[i]) for i in orden_diff]

fig_grupos = go.Figure()
fig_grupos.add_trace(go.Bar(
    y=labels_diff[::-1], x=imp_ab[orden_diff][::-1],
    orientation='h', name='Abandonan',
    marker_color='#e53e3e', opacity=0.85,
    hovertemplate='%{y}<br>Importancia SHAP: %{x:.4f}<extra>Abandonan</extra>',
))
fig_grupos.add_trace(go.Bar(
    y=labels_diff[::-1], x=imp_nab[orden_diff][::-1],
    orientation='h', name='No abandonan',
    marker_color='#3182ce', opacity=0.85,
    hovertemplate='%{y}<br>Importancia SHAP: %{x:.4f}<extra>No abandonan</extra>',
))
fig_grupos.update_layout(
    title='Importancia SHAP media por grupo — variables que más discriminan',
    barmode='group',
    xaxis=dict(title='Importancia media |SHAP|'),
    height=480, width=820,
    plot_bgcolor='#f7fafc', paper_bgcolor='white',
    legend=dict(orientation='h', y=1.08, x=0),
    margin=dict(l=200, r=40, t=80, b=40),
)
fig_grupos.show()
print('✅ Comparativa grupos generada.')

✅ Comparativa grupos generada.


In [8]:
# ============================================================
# CELDA 8: GENERAR HTML
# ============================================================
# Texto con el nombre del modelo ganador dinámico y bloque Wilcoxon.
# El HTML se guarda en docs/html/fase6/ y se deja una copia de respaldo
# en results/fase6/ — por eso se usa render_pagina_desde_fichero (devuelve
# el string) en lugar de render_pagina (que guardaría en un solo sitio).
# ============================================================
from src.html.wilcoxon_block import bloque_wilcoxon_html

def plotly_html(fig) -> str:
    return fig.to_html(full_html=False, include_plotlyjs='cdn')

def bloque(titulo: str, caption: str, html_fig: str) -> str:
    return (
        f'<div style="margin:32px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<div style="border-radius:8px;overflow:hidden">{html_fig}</div>'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        f'</div>'
    )

contenido = (
    '<h2 style="color:#2d3748">Fase 6 — Dashboard de Interpretabilidad SHAP</h2>'
    + bloque_wilcoxon_html(ROOT, nombre_ganador)
    + f'<p style="color:#4a5568;font-size:14px;max-width:900px;margin-bottom:8px">'
    f'Dashboard interactivo construido con Plotly sobre los valores SHAP del modelo '
    f'<strong>{nombre_ganador}</strong>. '
    f'SHAP (SHapley Additive exPlanations) permite explicar cada predicción individual: '
    f'cuánto contribuye cada variable a aumentar o reducir el riesgo de abandono.'
    '</p>'
    + f'<p style="color:#718096;font-size:12px;margin-bottom:32px">'
    f'Modelo: {nombre_ganador_pkl} | Test: {len(X_test):,} obs | '
    f'Features: {len(feature_names)}'
    '</p>'
    + bloque(
        '1. Waterfall — alumno detectado correctamente (TP)',
        'Cada barra muestra cuánto empuja una variable hacia el abandono (rojo) o en contra (azul). '
        'La suma da la probabilidad final predicha.',
        plotly_html(fig_wf_tp))
    + bloque(
        '2. Waterfall — alumno no detectado (FN)',
        'Falso negativo: el alumno abandona pero el modelo no lo detectó. '
        'Las variables protectoras superan a las de riesgo.',
        plotly_html(fig_wf_fn))
    + bloque(
        '3. Waterfall — falsa alarma (FP)',
        'Falso positivo: el modelo predice abandono pero el alumno no abandona.',
        plotly_html(fig_wf_fp))
    + bloque(
        '4. Beeswarm filtrable — top 12 variables',
        'Cada punto es un alumno. Usa los botones para filtrar por TP/FN/FP/TN. '
        'Permite ver si los patrones SHAP difieren entre grupos.',
        plotly_html(fig_bee))
    + bloque(
        '5. Top 200 alumnos de mayor riesgo',
        'Tabla ordenable con probabilidad predicha, clasificación y valores de las 5 variables '
        'más importantes. Verde=TP, rojo claro=FN, amarillo=FP, azul=TN.',
        plotly_html(fig_tabla))
    + bloque(
        '6. Importancia SHAP por grupo',
        'Comparativa entre alumnos que abandonan y no abandonan. '
        'Las variables con mayor diferencia son las más discriminantes.',
        plotly_html(fig_grupos))
)

html_completo = render_pagina_desde_fichero(
    'f6_m01d_shapash.ipynb',
    contenido,
    carpeta_notebook='fase6_evaluacion',
)
ruta_html = DIR_HTML / 'm01d_shapash.html'
ruta_html.write_text(html_completo, encoding='utf-8')

# Copia de respaldo
ruta_bak = DIR_RESULTS / 'shapash_dashboard.html'
ruta_bak.write_text(html_completo, encoding='utf-8')

print(f'✅ HTML generado: {ruta_html}')
print(f'   Respaldo:      {ruta_bak}')

✅ HTML generado: c:\PRUEBAS\AU_UJI_v2_RUTA_B\docs\html\fase6\m01d_shapash.html
   Respaldo:      c:\PRUEBAS\AU_UJI_v2_RUTA_B\results\fase6\shapash_dashboard.html


In [9]:
# ============================================================
# CELDA 9: NOTA — POR QUÉ NO SE USA SHAPASH EN MODO SERVIDOR
# ============================================================
# Shapash ofrece un dashboard Dash/Plotly con servidor local.
# No se implementa porque:
#   1. Requiere servidor local (puerto 8050)
#   2. No genera HTML embebible en la web del proyecto
#   3. En defensa de TFM en PC ajeno no funciona
#   4. Este dashboard Plotly ofrece las mismas funcionalidades
#      sin ninguna dependencia de servidor
#
# Para lanzarlo localmente en el futuro:
#   from shapash import SmartExplainer
#   xpl = SmartExplainer(model=modelo_ganador.named_steps['model'],
#                        features_dict=NOMBRES_LEGIBLES)
#   xpl.compile(x=X_test, y_pred=pd.Series(y_pred), y_target=pd.Series(y_true))
#   app = xpl.run_app(title_story='Abandono UJI — Shapash Dashboard')
# ============================================================
print('Nota Shapash modo servidor — ver comentarios de esta celda.')

Nota Shapash modo servidor — ver comentarios de esta celda.
